# Heart rate from your webcam, step by step

Each heartbeat pushes a bit more blood into the skin, which absorbs a bit more **green** light.
The face changes color by less than one gray level: invisible to us, but measurable.

1. Grab 20 s of webcam frames
2. Detect the face
3. Average the green channel of the face → one number per frame
4. Remove the slow drift
5. DFT: which frequencies are in that signal?
6. Strongest frequency between 36 and 180 bpm → heart rate

`main.py` does the same thing live.

In [ ]:
import time
from collections import deque

import cv2
import matplotlib.pyplot as plt
import numpy as np

SECONDS = 20
LOW_HZ, HIGH_HZ = 0.6, 3.0     # heart-rate band: 36 to 180 bpm
BOX_SMOOTH = 10                # face boxes averaged: ~0.33 s at 30 fps

## Step 1: grab 20 s of webcam frames

Sit still, with even light on your face. We show images with matplotlib instead of `cv2.imshow`,
which tends to freeze inside Jupyter.

In [ ]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
for _ in range(30):            # let the camera adjust its exposure first
    cap.read()

frames = []
start = time.time()
while time.time() - start < SECONDS:
    ok, frame = cap.read()
    frames.append(frame)
cap.release()                  # free the camera for other apps

fps = len(frames) / SECONDS
print(f"{len(frames)} frames in {SECONDS} s -> {fps:.1f} fps")

plt.imshow(cv2.cvtColor(frames[0], cv2.COLOR_BGR2RGB))   # OpenCV is BGR, matplotlib RGB
plt.axis("off")
plt.show()

A frame is just a 3D numpy array: **height × width × 3 color channels**, in the order **B, G, R**.

In [ ]:
frame = frames[0]
print("shape:", frame.shape, "| top-left pixel (B, G, R):", frame[0, 0])

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, name in enumerate(["blue", "green", "red"]):
    axes[i].imshow(frame[:, :, i], cmap="gray")
    axes[i].set_title(f"channel {i}: {name}")
    axes[i].axis("off")
plt.show()

## Step 2: detect the face

The Haar cascade is a classic face detector that ships with OpenCV 4.x.
It returns one `(x, y, w, h)` box per face.

In [ ]:
detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
faces = detector.detectMultiScale(gray, 1.1, 5, minSize=(80, 80))
print(faces)

x, y, w, h = max(faces, key=lambda f: f[2] * f[3])   # keep the largest face
img = frame.copy()
cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## Step 3: one number per frame

For every frame: find the face and average the green channel (index 1) of all the pixels inside the box.

The detector returns a slightly different box on every frame, and that jitter would leak into the
signal. So we use the **average of the last 10 boxes** instead: a `deque` with `maxlen` keeps only the
most recent ones. If a frame misses the face, the old boxes are simply reused.

In [ ]:
greens = []
boxes = deque(maxlen=BOX_SMOOTH)
for frame in frames:
    faces = detector.detectMultiScale(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 1.1, 5, minSize=(80, 80))
    if len(faces):
        boxes.append(max(faces, key=lambda f: f[2] * f[3]))
    x, y, w, h = np.mean(boxes, axis=0).astype(int)
    greens.append(frame[y:y+h, x:x+w, 1].mean())

greens = np.array(greens)
t = np.arange(len(greens)) / fps

plt.figure(figsize=(12, 3))
plt.plot(t, greens, color="green")
plt.xlabel("time (s)")
plt.ylabel("mean green")
plt.show()

Look at the y axis: the signal moves by only a few gray levels. The pulse is in there, hidden under
noise and slow changes in light. Our eyes can't find it, but the DFT can.

## Step 4: remove the slow drift

Light changes, auto exposure and small head movements make the signal wander slowly. Those slow
changes are much stronger than the pulse and would win in the DFT (you would almost always get the lowest value, 36 bpm).

A **moving average** over 1 s follows only the slow part of the signal. Subtracting it leaves the
fast part, where the pulse lives. (We subtract the mean first, because at the edges `np.convolve`
pads with zeros.)

In [ ]:
g = greens - greens.mean()
k = int(fps)                                          # 1 s worth of frames
slow = np.convolve(g, np.ones(k) / k, mode="same")    # moving average
pulse = g - slow

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax1.plot(t, g, color="green", label="signal")
ax1.plot(t, slow, color="black", label="1 s moving average")
ax1.legend()
ax2.plot(t, pulse, color="tab:red")
ax2.set_title("signal minus moving average")
ax2.set_xlabel("time (s)")
plt.show()

## Step 5: DFT

The DFT turns the signal into a list of frequencies and how strong each one is.
The Hann window fades the signal in and out at the edges so that whatever drift is left does not
leak into the heart-rate band. Then we only look at 0.6–3 Hz (36–180 bpm).

In [ ]:
spectrum = np.abs(np.fft.rfft(pulse * np.hanning(len(pulse))))
freqs = np.fft.rfftfreq(len(pulse), d=1 / fps)       # Hz of each DFT bin
band = (freqs >= LOW_HZ) & (freqs <= HIGH_HZ)

plt.figure(figsize=(12, 3))
plt.plot(freqs[band] * 60, spectrum[band])
plt.xlabel("beats per minute")
plt.ylabel("strength")
plt.show()

## Step 6: strongest frequency → heart rate

In [ ]:
bpm = freqs[band][np.argmax(spectrum[band])] * 60

plt.figure(figsize=(12, 3))
plt.plot(freqs[band] * 60, spectrum[band])
plt.axvline(bpm, color="red", linestyle="--")
plt.title(f"Heart rate: {bpm:.0f} bpm")
plt.xlabel("beats per minute")
plt.show()